# Module 9 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

## Naive Bayes Classifier

For this assignment you will be implementing and evaluating a Naive Bayes Classifier with the same data from last week:

http://archive.ics.uci.edu/ml/datasets/Mushroom

(You should have downloaded it).

<div style="background: lemonchiffon; margin:20px; padding: 20px;">
    <strong>Important</strong>
    <p>
        No Pandas. The only acceptable libraries in this class are those contained in the `environment.yml`. No OOP, either. You can used Dicts, NamedTuples, etc. as your abstract data type (ADT) for the the tree and nodes.
    </p>
</div>


You'll first need to calculate all of the necessary probabilities using a `train` function. A flag will control whether or not you use "+1 Smoothing" or not. You'll then need to have a `classify` function that takes your probabilities, a List of instances (possibly a list of 1) and returns a List of Tuples. Each Tuple has the best class in the first position and a dict with a key for every possible class label and the associated *normalized* probability. For example, if we have given the `classify` function a list of 2 observations, we would get the following back:

```
[("e", {"e": 0.98, "p": 0.02}), ("p", {"e": 0.34, "p": 0.66})]
```

when calculating the error rate of your classifier, you should pick the class label with the highest probability; you can write a simple function that takes the Dict and returns that class label.

As a reminder, the Naive Bayes Classifier generates the *unnormalized* probabilities from the numerator of Bayes Rule:

$$P(C|A) \propto P(A|C)P(C)$$

where C is the class and A are the attributes (data). Since the normalizer of Bayes Rule is the *sum* of all possible numerators and you have to calculate them all, the normalizer is just the sum of the probabilities.

You will have the same basic functions as the last module's assignment and some of them can be reused or at least repurposed.

`train` takes training_data and returns a Naive Bayes Classifier (NBC) as a data structure. There are many options including namedtuples and just plain old nested dictionaries. **No OOP**.

```
def train(training_data, smoothing=True):
   # returns the Decision Tree.
```

The `smoothing` value defaults to True. You should handle both cases.

`classify` takes a NBC produced from the function above and applies it to labeled data (like the test set) or unlabeled data (like some new data). (This is not the same `classify` as the pseudocode which classifies only one instance at a time; it can call it though).

```
def classify(nbc, observations, labeled=True):
    # returns a list of tuples, the argmax and the raw data as per the pseudocode.
```

`evaluate` takes a data set with labels (like the training set or test set) and the classification result and calculates the classification error rate:

$$error\_rate=\frac{errors}{n}$$

Do not use anything else as evaluation metric or the submission will be deemed incomplete, ie, an "F". (Hint: accuracy rate is not the error rate!).

`cross_validate` takes the data and uses 10 fold cross validation (from Module 3!) to `train`, `classify`, and `evaluate`. **Remember to shuffle your data before you create your folds**. I leave the exact signature of `cross_validate` to you but you should write it so that you can use it with *any* `classify` function of the same form (using higher order functions and partial application). If you did so last time, you can reuse it for this assignment.

Following Module 3's discussion, `cross_validate` should print out the fold number and the evaluation metric (error rate) for each fold and then the average value (and the variance). What you are looking for here is a consistent evaluation metric cross the folds. You should print the error rates in terms of percents (ie, multiply the error rate by 100 and add "%" to the end).

To summarize...

Apply the Naive Bayes Classifier algorithm to the Mushroom data set using 10 fold cross validation and the error rate as the evaluation metric. You will do this *twice*. Once with smoothing=True and once with smoothing=False. You should follow up with a brief explanation for the similarities or differences in the results.

In [1]:
from copy import deepcopy
import numpy as np
import random
from typing import List, Dict, Tuple, Callable

In [2]:
def create_folds(xs: List, n: int) -> List[List[List]]:
    np.random.shuffle(xs)
    k, m = divmod(len(xs), n)
    # be careful of generators...
    return list(xs[i * k + min(i, m):(i + 1) * k + min(i + 1, m)] for i in range(n))

In [3]:
def create_train_test(folds: List[List[List]], index: int) -> Tuple[List[List], List[List]]:
    training = []
    test = []
    for i, fold in enumerate(folds):
        if i == index:
            test = fold
        else:
            training = training + fold.tolist()
    return np.array(training), np.array(test)

<a id="train"></a>
## train

The Naive Bayes classifier is a probabilistic Naive Bayes classifier. It assumes conditional independence between every pair of features, given the value of the class variable. Based on the Bayes' Theorem, the classifier calculates the probability of an event, based on prior knowledge of the conditions of the training set. This is expressed as the following: 

$$ P(C|X) = \frac{P(X|C)P(C)}{P(X)} $$

The prior probability, $P(C)$, is the probability of finding the class label, $C$ in the training set. The conditional probability, $P(X|C)$, is the probability of a feature value, $X$ occuring given a specific class label, $C$. $P(X)$ is the prior probability of the predictors; however, this denominator is not required to calculate $P(C|X)$. 

In the Naive Bayes algorithm, the training of the model is the calculation of prior probabilities and conditional probabilites required to allow for quick calculation of $P(C|X)$ to make an informed prediction. All the required probabilities are calculated for each feature/feature value pair, given each class label. 

* **data** np.array: dataset where the first column of the array is the label
* **smoothing** Boolean: whethere Laplace's smoothing should be applied to calculation of the feature probability
  
**returns** dict, dict: the calculated prior class probabilities and the conditional feature probabilities stored in its respective dictionaries

In [4]:
def train(training_data, smoothing=True):
    class_count = {}
    feature_count = {}
    class_probabilities = {}
    feature_class_probabilities = {}

    #count occurences in the dataset
    for instance in training_data:
        label = instance[0]
        features = instance[1:]

        #class label occurences
        class_count[label] = class_count.get(label, 0) + 1

        #feature occurences given a class label
        for feature_index, feature_value in enumerate(features):
            if (feature_index, feature_value, label) not in feature_count:
                feature_count[(feature_index, feature_value, label)] = 1
            else:
                feature_count[(feature_index, feature_value, label)] += 1

    #calculating class probabilities
    total_count = len(training_data)
    for class_label in class_count:
        class_probabilities[class_label] = class_count[class_label] / total_count

    #calculating conditional proababilities
    for (feature_index, feature_value, class_label), count in feature_count.items():
        if smoothing: 
            feature_class_probabilities[(feature_index, feature_value, class_label)] = (count + 1) / (class_count[class_label] + len(set([fp[1] for fp in feature_count if fp[2] == class_label])))

        else: 
            feature_class_probabilities[(feature_index, feature_value, class_label)] = count / class_count[class_label]

    return class_probabilities, feature_class_probabilities

In [5]:
test_data = [
    ['a', 'e', 'f'],
    ['a', 'd', 'f'],
    ['b', 't', 'c'],
    ['b', 't', 'c']
]
test_class, feature_class_probabilities = train(test_data, smoothing=True)
assert test_class == {'a': 0.5, 'b': 0.5}
assert feature_class_probabilities == {(0, 'e', 'a'): 0.4, (1, 'f', 'a'): 0.6, (0, 'd', 'a'): 0.4, (0, 't', 'b'): 0.75, (1, 'c', 'b'): 0.75}

test_class, feature_class_probabilities = train(test_data, smoothing=False)
assert len(test_class) == 2
assert feature_class_probabilities == {(0, 'e', 'a'): 0.5, (1, 'f', 'a'): 1.0, (0, 'd', 'a'): 0.5, (0, 't', 'b'): 1.0, (1, 'c', 'b'): 1.0}


<a id="classify"></a>
## classify

Once the probabilities of the dataset are calculated, classification is done by calculating the probability of each class label occuring, given the features of the data point. The label with the highest calculated probability is assigned to the datapoint as a prediction.

$$ label = argmax_{c}P(X|C)P(C) $$

This function calculates the probabilty of each possible class label for each data point and assigns the label of highest probability to the data point. 

* **trained_model** tuple(dict, dict): prior class probabilites and conditional feature probabilities stored in dictionaries
* **observations** np.array: test data to be classified
* **labeled** boolean: true if the dataset is labelled
  
**returns** list: list of class label predictions assigned to each instance in the test dataset

In [6]:
def classify(trained_model, observations, labeled=True):
    class_probabilities, feature_probabilities = trained_model
    predictions = []
    
    for i in observations:
        features = i[1:] if labeled else i
        scores = {}
        
        for label in class_probabilities:
            class_score = class_probabilities[label]
            for feature_index, feature_value in enumerate(features):
                class_score *= feature_probabilities.get((feature_index, feature_value, label), 0)
            scores[label] = class_score

        total_score = sum(scores.values())
        normalized_scores = {label: score / total_score for label, score in scores.items()}

        best_class = max(normalized_scores, key = normalized_scores.get)
        predictions.append((best_class, normalized_scores))

    return predictions

In [7]:
test_class = {'a': 0.5, 'b': 0.5}
feature_class_probabilities = {(0, 'e', 'a'): 0.4, (1, 'f', 'a'): 0.6, (0, 'd', 'a'): 0.4, (0, 't', 'b'): 0.75, (1, 'c', 'b'): 0.75}

observation = [['a', 'e', 'f']]
test_predictions = classify((test_class, feature_class_probabilities), observation, labeled=True)
assert len(test_predictions) == 1
assert test_predictions[0][0] == 'a'
assert len(test_predictions[0][1]) == 2

<a id="evaluate"></a>
## evaluate

Once the prediction classifications are made for a test dataset, the model should be evaluated for performance. The error rate of classification is a metric used to evaluate the efficiency of the model. The error rate can be calculated with the following formula:

$$error\_rate=\frac{errors}{n}$$

This function calculates the error rate of the classification predictions made on a dataset.

* **actuals** list: actual labels of the dataset
* **predictions** list: predicted labels of the dataset

  
**returns** float: calculated error rate of the predictions

In [8]:
def evaluate(actuals, predictions):
    total = len(actuals)
    errors = sum(1 for actual, predicted in zip(actuals, predictions) if actual != predicted)
    error_rate = errors/total
    return round(error_rate, 4)

In [9]:
actual = ['a', 'b', 'c']
prediction = ['a', 'b', 'c']
assert evaluate(actual, prediction) == 0.0

actual = ['a', 'c', 'c', 'd']
prediction = ['a', 'b', 'b', 'd']

assert evaluate(actual, prediction) == 0.5

actual = ['d', 'c', 'c', 'a']
prediction = ['a', 'b', 'b', 'd']
assert evaluate(actual, prediction) == 1.0

<a id="cross_validate"></a>
## cross_validate

Cross validation is a resampling method used to effectively evaluate machine learning models when data availability is limited. The dataset is shuffled and divided into n number of folds. Iteratively, each fold is taken as the test set, while the remaining folds are taken as the training set. The folds are iterated through and fitted and run through the model, n number of times, resulting in n evaluations of the model. 

This function generates 10 folds, iterates through the list of folds and applies the ID3 decision tree algorithm to train and construct a decision tree and predict the test set with it. The error rate is calculated for each fold and averaged across all folds. 

* **data** np.array: dataset where the first column of the array is the label
* **labeled** Boolean: if the dataset is labeled, or not
* **smoothing** Boolean: whethere Laplace's smoothing should be applied to calculation of the feature probability
  
**returns** float: average error rate across all folds

In [10]:
def cross_validate(data, labeled = True, smoothing = True):
    folds = create_folds(data, 10)

    fold_error, test_error, train_error = [], [], []
    
    for i in range(len(folds)):
        train_data, test_data = create_train_test(folds, i)

        trained_model = train(train_data, smoothing)
        train_predictions = classify(trained_model, train_data, labeled)
        test_predictions = classify(trained_model, test_data, labeled)

        train_error_rate = evaluate(train_data.T[0], [t[0] for t in train_predictions])
        test_error_rate = evaluate(test_data.T[0], [t[0] for t in test_predictions])

        fold_error.append([i, test_error_rate, train_error_rate])
        test_error.append(test_error_rate)
        train_error.append(train_error_rate)
    
    test_average = sum(test_error)/len(test_error)
    train_average = sum(train_error)/len(train_error)
    test_std = (sum([((x - test_average) ** 2) for x in test_error]) / len(test_error)) ** 0.5
    train_std = (sum([((x - train_average) ** 2) for x in train_error]) / len(train_error)) ** 0.5

    print ("{:<8} {:<10} {:<10}".format('Fold','Test','Train'))
    for list in fold_error:
        print("{:<8} {:<10} {:<10}".format(list[0],list[1],list[2]))
    print ("{:<8} {:<10} {:<10}".format('Average', round(test_average, 4), round(train_average, 4)))
    print ("{:<8} {:<10} {:<10}".format('STD', round(test_std, 4), round(train_std,4))) 

    return round(test_average, 4)

In [11]:
test_data = np.array([
    [1, 2, 3],
    [1, 2, 3],
    [1, 2, 3],
    [1, 2, 3],
    [1, 2, 3],
    [1, 2, 3]
])
folds = create_folds(test_data, 3)
assert len(folds) == 3

test_error = [1, 2, 4, 5, 6]
test_average = sum(test_error)/len(test_error) 
assert test_average == 3.6

test_std = (sum([((x - test_average) ** 2) for x in test_error]) / len(test_error)) ** 0.5
assert 1.85 < test_std < 1.86

## Dataset

In [12]:
#data loading and cleaning
data = np.loadtxt('agaricus-lepiota.data', delimiter=",", dtype= 'str')

data_cleaned = data[~(data[:,11] == '?'),:] #missing values only found in 11th feature (12th column)

assert len(data) - len(data_cleaned) == 2480  #number of missing values according to dataset information

In [13]:
# run with smoothing
mean_error = cross_validate(data_cleaned, True, True)

Fold     Test       Train     
0        0.0035     0.0024    
1        0.0035     0.0028    
2        0.0        0.003     
3        0.0018     0.0028    
4        0.0018     0.0028    
5        0.0035     0.0028    
6        0.0035     0.0026    
7        0.0053     0.0026    
8        0.0018     0.0028    
9        0.0035     0.0028    
Average  0.0028     0.0027    
STD      0.0014     0.0002    


In [14]:
#run without smoothing
mean_error = cross_validate(data_cleaned, True, False)

Fold     Test       Train     
0        0.0035     0.0028    
1        0.0018     0.0028    
2        0.0035     0.0024    
3        0.0018     0.0026    
4        0.0035     0.0028    
5        0.0018     0.003     
6        0.0018     0.003     
7        0.0        0.0028    
8        0.0035     0.0026    
9        0.0071     0.0024    
Average  0.0028     0.0027    
STD      0.0018     0.0002    


Overall, a significant difference was not found between the Naive Bayes algorithm with and without smoothing. This usually occurs when the dataset is rich enough that each feature's unique values appear in each fold during training. When the dataset has low sparsity as such, smoothing has a minimal effect. Another reason for low difference between smoothing and non-smoothing approaches is feature relevance. If many features do not contribute to the variance between classes, smoothing of such features would not impact overall accuracy. 

## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.